# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
!python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 746.6/746.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 895.2/895.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.6/433.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.1 MB/s eta 0:00:00


## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial.

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [11]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name= "langchain-multi-query-demo", #give it a meaningful name,
    embedding=embeddings,
)

Elastic Cloud ID: ··········
Elastic Api Key: ··········
OpenAI API key: ··········


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [12]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [17]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

def metadata_func(record: dict, metadata: dict) -> dict:
    # Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    filtered_metadata = {key: value for key, value in record.items() if value is not None}

    return filtered_metadata

# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=50  # Define chunk size and chunk overlap
)

docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [21]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="langchain-multi-query-demo"
,
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [22]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible.

    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team operate within NASA?', '3. What are the responsibilities of the NASA sales team?']


---- Answer ----
The NASA sales team is a part of the Americas region in the sales organization of the company. It is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. The team is responsible for promoting and selling the company's products and services in the North and South American markets. They work closely with other departments, such as marketing, product development, and customer support, to ensure customer satisfaction and achieve the company's business objectives.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [23]:
# Update index name for a different dataset
index_name = "advanced-query-retrieval"

documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name=index_name,  # Updated index name
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0.5, openai_api_key=OPENAI_API_KEY)  # Adjust temperature for varied responses

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(search_kwargs={"k": 5}), llm)  # Retrieve top 5 results

In [25]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document
import logging

# Logging setup for debugging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# Define the prompt for LLM response generation
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an AI assistant for answering questions. Utilize the following retrieved context to formulate a detailed response.

    If the answer is unknown, simply state that. Be clear, educational, and thorough.

    Context: {context}
    Question: "{question}"
    Answer:
    """
)

# Formatting retrieved documents for better readability
LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """---
SOURCE: {name}
{page_content}
---
"""
)

# Function to format retrieved documents
def _combine_documents(docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"):
    formatted_docs = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(formatted_docs)

# Integrate the retriever into a parallel processing pipeline
_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

# Chain the context retrieval with LLM response generation
chain = _context | LLM_CONTEXT_PROMPT | llm

# Invoke the chain for query execution
ans = chain.invoke("What are NASA’s latest technological advancements?")

print("---- AI Response ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the most recent technological breakthroughs made by NASA?', "2. Can you provide information on NASA's most recent advancements in technology?", '3. What are the cutting-edge technologies currently being developed by NASA?']


---- AI Response ----
I am an AI assistant and do not have access to current information on NASA's technological advancements. However, I can provide information on the sales organization structure of the company, their sales strategy for fiscal year 2024, the importance of collaboration between sales and engineering teams, and the career matrix for software engineers. If you would like information on NASA's latest technological advancements, I suggest visiting their official website or contacting their media relations team for the most up-to-date information.


In [24]:
# Define new index name for chunked embeddings storage
index_name = "semantic-chunked-query"

documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name=index_name,  # New index name
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

# Modify LLM parameters for controlled response behavior
llm = OpenAI(temperature=0.3, openai_api_key=OPENAI_API_KEY)

# Chunk-based retriever with optimized multi-query retrieval
retriever = MultiQueryRetriever.from_llm(
    vectorstore.as_retriever(search_kwargs={"k": 3, "score_threshold": 0.7}), llm
)  # Retrieve top 3 relevant results with a confidence threshold

In [26]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document
import logging

# Logging setup for debugging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# Define a refined prompt for context-based AI responses
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an advanced AI assistant for question-answering tasks. Use the retrieved context to generate a well-structured response.

    If the answer is unknown, state that clearly. Ensure educational and detailed insights.

    Context: {context}
    Question: "{question}"
    Answer:
    """
)

# Formatting retrieved documents for better readability
LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """---
SOURCE: {name}
{page_content}
---
"""
)

# Function to format retrieved documents while filtering irrelevant results
def _combine_documents(docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"):
    # Filter documents with low relevance scores (if applicable)
    filtered_docs = [doc for doc in docs if doc.metadata.get("score", 1) >= 0.7]
    formatted_docs = [format_document(doc, document_prompt) for doc in filtered_docs]
    return document_separator.join(formatted_docs)

# Multi-query retrieval pipeline with improved document processing
_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

# Chain the retriever and AI response generation
chain = _context | LLM_CONTEXT_PROMPT | llm

# Invoke the chain for query execution
ans = chain.invoke("How is semantic search improving AI applications?")

print("---- AI Response ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the benefits of incorporating semantic search into AI applications?', '2. In what ways does semantic search enhance the performance of AI applications?', '3. Can you explain the impact of semantic search on improving AI applications?']


---- AI Response ----

Semantic search is improving AI applications by allowing them to understand the context and meaning behind words and phrases, rather than just matching keywords. This enables AI to retrieve more accurate and relevant information, leading to better question-answering capabilities and more personalized responses. Additionally, semantic search can help AI assistants better understand and interpret human language, making them more user-friendly and efficient.
